# 连乘与对数

语言模型在计算回答 y 的概率时，是一个 token 一个 token 地生成的。假设回答 y 由 T 个 token 组成：

y = (y_1, y_2, …, y_T)

那么整段回答的条件概率可以写成：

π_θ(y | x) = rod_{t=1}^T π_θ(y_t | x, y_{<t})

其中：

- π_θ(y | x)：参数为 θ 的模型，在 prompt x 后生成整段回答 y 的概率。
- y_t：回答中的第 t 个 token。
- y_{<t}：第 t 个 token 之前已经生成的所有 token。
- ∏：连乘符号，把每一步 token 的概率相乘。

实际代码中几乎不会直接乘概率——多个小于 1 的数连乘会导致数值下溢。因此通常取对数，将连乘变成求和：

log π_θ(y | x) = um_{t=1}^T log π_θ(y_t | x, y_{<t})

这就是 sequence_logprob 的原理：先对每个 token 取 log probability，再只把回答部分的 token log probability 加起来。

# PPO目标函数推理

**核心结论**

DPO 的推导起点不是 PPO-Clip 损失，而是一个带 KL 约束的 RLHF 目标：既希望模型生成高奖励回答，又不希望模型偏离冻结的参考模型太远。PPO 只是传统 RLHF 中用来近似优化这个目标的一种算法。

**1. 只追求奖励会有什么问题？**

最直接的目标是：

$$
\max_{\pi}\;\mathbb{E}_{x\sim\mathcal D,\;y\sim\pi(\cdot\mid x)}[r(x,y)]
$$

其中，$r(x,y)$ 是奖励模型对回答 $y$ 的评分。如果只最大化奖励，策略可能为了骗取奖励而生成奇怪文本，并丢失原有的语言能力。

**2. 加入“不许偏离参考模型太远”的硬约束**

设 $\pi_{\mathrm{ref}}$ 是冻结的 SFT 参考模型，可以写成：

$$
\max_{\pi}\;\mathbb{E}[r(x,y)]
$$

满足：

$$
D_{\mathrm{KL}}\!\left(\pi(\cdot\mid x)\,\Vert\,\pi_{\mathrm{ref}}(\cdot\mid x)\right)\leq\delta
$$

$\delta$ 是允许的最大偏离程度。KL 越小，新策略越接近参考模型；KL 越大，新策略变化越明显。

**3. 用拉格朗日乘子把硬约束写进目标函数**

引入非负乘子 $\beta\geq 0$：

$$
\max_{\pi}
\left[
\mathbb{E}[r(x,y)]
-
\beta\left(
D_{\mathrm{KL}}(\pi\Vert\pi_{\mathrm{ref}})-\delta
\right)
\right]
$$

这里的 $D_{\mathrm{KL}}-\delta$ 表示超出限制多少。如果 KL 超过 $\delta$，这一项为正，前面的负号就会降低总目标。$\beta$ 可以理解为“偏离参考模型的价格”。

展开括号：

$$
\max_{\pi}
\left[
\mathbb{E}[r(x,y)]
-
\beta D_{\mathrm{KL}}(\pi\Vert\pi_{\mathrm{ref}})
+
\beta\delta
\right]
$$

在优化策略 $\pi$ 时，$\beta$ 和 $\delta$ 是固定值，因此 $\beta\delta$ 对所有候选策略都相同。给所有策略的分数加同一个常数不会改变谁最大，所以可以去掉它：

$$
\boxed{
\max_{\pi}
\left[
\mathbb{E}[r(x,y)]
-
\beta D_{\mathrm{KL}}(\pi\Vert\pi_{\mathrm{ref}})
\right]
}
$$

这不是忽略约束，而是把约束变成了偏离策略时需要支付的代价。在适当条件下，可以找到一个 $\beta$，使惩罚形式与原来的硬约束问题具有相同的最优解。

**4. $\beta$ 的直觉**

- $\beta$ 较大：偏离代价高，策略更加保守，更接近参考模型。
- $\beta$ 较小：偏离代价低，策略更愿意为了奖励改变行为。
- $\beta=0$：完全不限制策略偏离，只追求奖励。

因此可以把最终目标记成：

> 最终目标 = 回答奖励 − 偏离参考模型的代价

**5. 它和 PPO 到底是什么关系？**

上面的公式描述的是 RLHF 最终想优化什么；PPO 描述的是怎样稳定地更新神经网络。PPO 真正使用的核心是裁剪代理目标：

$$
L^{\mathrm{PPO}}(\theta)
=
\mathbb{E}_t\!\left[
\min\!\left(
\rho_t(\theta)\hat A_t,
\operatorname{clip}(\rho_t(\theta),1-\epsilon,1+\epsilon)\hat A_t
\right)
\right]
$$

其中：

$$
\rho_t(\theta)
=
\frac{\pi_\theta(a_t\mid s_t)}{\pi_{\mathrm{old}}(a_t\mid s_t)}
$$

需要区分两个模型：

- $\pi_{\mathrm{ref}}$：冻结的 SFT 参考模型，限制整个 RLHF 训练不要偏离原模型太远。
- $\pi_{\mathrm{old}}$：PPO 上一轮策略，限制当前一次参数更新不要变化过大。

**一句话总结**

DPO 从“奖励最大化 + KL 偏离限制”的 RLHF 目标出发，通过数学推导把显式奖励函数消掉；PPO 则是传统 RLHF 用来优化该目标的策略梯度算法，二者不要混为同一个目标函数。
